# SQL Databases


## Create a dimple SQLite Database


In [7]:
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [2]:
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

In [3]:
# Create Table
cursor.execute(
  '''CREATE TABLE IF NOT EXISTS employees(
    id INTEGER PRIMARY KEY,
    name TEXT,
    role TEXT,
    department TEXT,
    salary REAL
    )'''
)

In [4]:
cursor.execute(
  '''CREATE TABLE IF NOT EXISTS projects(
    id INTEGER PRIMARY KEY,
    name TEXT,
    status TEXT,
    budget REAL,
    lead_id INTEGER)'''
)

In [6]:
# Data sample
employees = [
    (1, "Alice Smith", "Software Engineer", "Engineering", 90000),
    (2, "Bob Johnson", "Data Scientist", "Data Science", 95000),
    (3, "Charlie Brown", "Product Manager", "Product", 105000),
    (4, "Diana Prince", "UX Designer", "Design", 85000),
    (5, "Ethan Hunt", "DevOps Engineer", "Engineering", 92000)
]

projects = [
    (1, "Project Alpha", "Active", 150000, 1),
    (2, "Project Beta", "Completed", 200000, 2),
    (3, "Project Gamma", "Active", 120000, 3),
    (4, "Project Delta", "On Hold", 180000, 4),
    (5, "Project Epsilon", "Active", 160000, 1)
]

In [7]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)', employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?)', projects)

In [9]:
cursor.execute("SELECT * FROM employees")

In [10]:
conn.commit()
conn.close()

# Database Content Extraction

In [1]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

/home/valac/Projects/RAG/RAG-Udemi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Method 1: SQL Database Utility
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

## Get database info
print(f"Tables {db.get_usable_table_names()}")
print(f"\nTable DDL") # Data Definition Language
print(db.get_table_info())

Tables ['employees', 'projects']

Table DDL

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Alice Smith	Software Engineer	Engineering	90000.0
2	Bob Johnson	Data Scientist	Data Science	95000.0
3	Charlie Brown	Product Manager	Product	105000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	Project Alpha	Active	150000.0	1
2	Project Beta	Completed	200000.0	2
3	Project Gamma	Active	120000.0	3
*/


In [13]:
from typing import List
from langchain_core.documents import Document

# Method 2: Custom SQL to Document Conversion
print(f"Custom SQL Processing\n")
def sql_to_documents(db_path: str) -> List[Document]:
    """Convert SQL Database to documents with context"""
    # create a connection to the database
    conn = sqlite3.connect(db_path)
    # create a cursor object to execute SQL commands
    cursor = conn.cursor()
    # get the list of tables in the database
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    # fetch all table names  NOTE: Fetch in Spanish means "obtener"
    tables = cursor.fetchall()
    # initialize an empty list to store documents
    documents =[]

    # Strategy 1: Create documents for each table
    for table in tables:
        table_name = table[0]

        # get the table schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        # this creates a list of column names using list comprehension 
        # where col[1] is the second element of each column tuple which contains the column name, 
        # we don't use column[0] because that is the column id
        column_names = [col[1] for col in columns] 

        # get table data
        cursor.execute(f"SELECT * from {table_name}")
        rows = cursor.fetchall()
        
        # Create a table overview document with schema and sample data
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {' ,'.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"


        # Add sample records to the document
        table_content += "Sample Records:\n"
        for row in rows[:5]:  # Add first 5 records as sample
            # Create a dictionary of column names and values, 
            # zip is used to pair each column name with its corresponding value in the row
            record = dict(zip(column_names, row)) 
            table_content += f"{record}\n\n"

        doc = Document(
            page_content = table_content,
            metadata = {
                'source': db_path,
                'table_name': table_name,
                'num_records': len(rows),
                'data_type': 'sql_table'
            }
        )

        # Append the document to the list
        documents.append(doc)

    # Strategy 2: Create relationship documents
    cursor.execute("""
        SELECT e.name, e.role, p.name as project_name, p.status
        FROM employees e
        JOIN projects p on e.id = p.lead_id
    """)
    
    relationships = cursor.fetchall()
    rel_content = "Employee-Project Relationships:\n\n"
    for rel in relationships:
        rel_content += f"Employee: {rel[0]}-{rel[1]}-, leads: {rel[2]} - Status: {rel[3]}\n"
    
    rel_doc = Document(
        page_content = rel_content,
        metadata = {
            'source': db_path,
            'data_type': 'sql_relationships',
            'query': 'employee_project_join'
        }
    )
    documents.append(rel_doc)
    return documents




Custom SQL Processing



In [14]:
sql_to_documents('data/databases/company.db')

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id ,name ,role ,department ,salary\nTotal Records: 5\n\nSample Records:\n{'id': 1, 'name': 'Alice Smith', 'role': 'Software Engineer', 'department': 'Engineering', 'salary': 90000.0}\n\n{'id': 2, 'name': 'Bob Johnson', 'role': 'Data Scientist', 'department': 'Data Science', 'salary': 95000.0}\n\n{'id': 3, 'name': 'Charlie Brown', 'role': 'Product Manager', 'department': 'Product', 'salary': 105000.0}\n\n{'id': 4, 'name': 'Diana Prince', 'role': 'UX Designer', 'department': 'Design', 'salary': 85000.0}\n\n{'id': 5, 'name': 'Ethan Hunt', 'role': 'DevOps Engineer', 'department': 'Engineering', 'salary': 92000.0}\n\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: projects\nColumns: id ,name ,status ,budget ,lead_id\